### Langgrpah API

In [1]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    messages:Annotated[list, add_messages]


graph_builder = StateGraph(State)

graph_builder


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash")
response = llm.invoke("Sing a ballad of LangChain.")
print(response)


In [ ]:
def chatbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

graph_builder.add_node("llmchatbot", chatbot)
graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

In [ ]:
graph_builder.compile()

In [ ]:
graph = graph_builder.compile()
result = graph.invoke({"messages": [{"role": "user", "content": "hi"}]})
result

In [ ]:
result["messages"][-1].content

In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)
tool.invoke("what is langgraph")

In [ ]:
def multiply(a:int, b:int) -> int:
    """Multiply two integers."""
    return a * b

In [ ]:
tools=[tool, multiply]
llm_with_tools = llm.bind_tools(tools)
llm_with_tools


In [ ]:
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition


from langgraph.checkpoint.memory import MemorySaver

memory_saver = MemorySaver()

def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", 
                               tools_condition)
builder.add_edge("tools", "tool_calling_llm")
builder.add_edge("tool_calling_llm", END)

graph= builder.compile(checkpointer=memory_saver)
graph

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "what is 2 * 4"}]})

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "current AMZN stock price"}]})

In [ ]:
response = graph.invoke({"messages": [{"role": "user", "content": "waht is current BLK stock price and what is 2 * 4"}]})
for m in response["messages"]:
    m.pretty_print()

In [ ]:
from langchain_core.runnables import RunnableConfig


config: RunnableConfig = {"configurable": {"thread_id": "t1"}}

response = graph.invoke({"messages": [{"role": "user", "content": "my name is kurasaki"}]}, config=config)
for m in response["messages"]:
    m.pretty_print()

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "waht is my name ?"}]}, config=config)

In [4]:
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash")
response = llm.invoke("Sing a ballad of LangChain.")
print(response)    

class State(TypedDict):
    messages:Annotated[list, add_messages]

def super_bot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

graph_builder_1 = StateGraph(State)
graph_builder_1.add_node("super_bot", super_bot)
graph_builder_1.add_edge(START, "super_bot")
graph_builder_1.add_edge("super_bot", END)

checkpoint_memory = MemorySaver()

graph_1 = graph_builder_1.compile(checkpointer=checkpoint_memory)

config: RunnableConfig = {"configurable": {"thread_id": "t2"}}
response = graph_1.invoke({"messages": [{"role": "user", "content": "what is my name ?"}]}, config=config)
for m in response["messages"]:
    m.pretty_print()


content=[{'type': 'text', 'text': 'Come gather round, ye hackers bold,\nAnd listen to my tune,\nOf silicon gods with minds of gold\nBeneath a digital moon.\nThey spoke in tongue of every sage,\nYet wandered blind and cold—\nThey had no hands to turn the page,\nNo memory to hold.\n\nFor stateless were the ancient beasts,\nThough vast their token’s breath;\nA prompt would give them life a while,\nThe next would bring their death.\n"Who’ll build a bridge?" the coders cried,\n"Across this wild domain?\nWho’ll tame the prompt-craft’s stormy tide\nAnd harness up the brain?"\n\nThen stepped a smith into the fray,\nYoung Harrison his name,\nTo forge a harness from the void\nAnd give the ghost a frame.\nHe gathered up the python threads,\nThe parser and the pipe,\nTo bind the monster’s wandering dread\nAnd make the logic ripe.\n\n*Oh, strike the key and link the chain,*\n*From prompt to parse and back again!*\n*Through vector seas where embeddings float,*\n*He set the wild machine afloat!*\n\nH

In [6]:
config: RunnableConfig = {"configurable": {"thread_id": "t3"}}

for chunk in graph_1.stream({"messages": [{"role": "user", "content": "what is my name ?"}]}, config=config, stream_mode="updates"):
    print(chunk)

{'super_bot': {'messages': [AIMessage(content=[{'type': 'text', 'text': "I still don't know your name! As an AI, I don't have access to your personal information or account details. \n\nIf you'd like, you can tell me your name, and I'll remember it for the rest of our conversation.", 'extras': {'signature': 'Er4GCrsGAWkUfRPeeqOpBh07c0272DDclPUAOEB8B4Vv5qLbbevIUTIenUUnqJvT2ORwALJ88ovWDdCEUOz6uvnMo8bkplrSmhzVP7MEFYjrwfE6dH1LfeHOBQSHFUCw+5mhfC5He4XAaU4lcDjxU9EXOTagC/x/B7omelDvqPdreWfJMaUeVq7kGnnSEr752zkVOESyCdTzKt1L98hlkcUXqvkULNBbmK/Xnn3j/BvV1HCNzP2HKushuWGjMKtRCb+zr5nlyqHjiR1N1aYklfWFzgLajzMJdWSA6YNNOzVE7WTqrNPcVoXOVxkt6zwPfw17rdz9qwQujyERg37Eb8LMSU5/ZQiCjHr+jv7FAWYSBvztw7nlmtimBzyKHEvujq6Pa9bAwse/loKtZJYw0gah0r0iDQHrQwsNtLRGOyE4FqfUUW0cySJWiMypJhyvJrlksCqIQNb9ZW+mgpZQvzPFKE4Gx2kj6mOesA5jqBpkhcqZ9BFw/+Wnzl3En8OUvJR+yfQxP2Yhl9i6sjHGMf4B1Ft7ovhJ0JfNtVyICrgDlsNmTy12+E466KwGxFidO6Z8buaFutUWarEh4qQgd7drmYsWAP0sfR/hPzYTzkyJ7uxvtrToEXAOQ9PIlRJReL1pImnyIEmzMpqHodd8K1BOWzTLFZhUaWpnTVsgu9NN/kfhmC

In [7]:
for chunk in graph_1.stream({"messages": [{"role": "user", "content": "my bname is krish"}]}, config=config, stream_mode="values"):
    print(chunk)

{'messages': [HumanMessage(content='what is my name ?', additional_kwargs={}, response_metadata={}, id='6283bba0-6142-4f30-878f-72dd8b67132d'), AIMessage(content=[{'type': 'text', 'text': "I don't know your name because you haven't told me yet! What should I call you?", 'extras': {'signature': 'EqkGCqYGAWkUfROeD4KVIutN5l3v+ngjRX5ag0xNNEB4Cd1pokIFTJ5xinC4JOb0z1DZrfMgQvO9sW4GMt8k4QhGnFPYoJpCaCpOl2yFyvnGp30HoXChSIIKwnlb19uzILcw35ActEZeJhZJwtB5X8MVGxsTQ3IjwacGm3yH9vXP1XCnzxS69jCV8Ec19FLNVL1D2AcepZM1q1Dj1GWnRrohW+41Tv2pOads6jeyMET6y7YzTBSjNSyJUqLjEWgIVy7pQPQMiD1M4TeNqOJkL3kyBbiHN30OIeNRqIG89mRUu91eP+922DnsCuK+kOsU7S3eT9Y10NA6ae+nSGq+iFiBnWjRgwiJ4qYBksChHrd9FPaKmDGbd8vHLyiOjVlWtkUhQ1MclrZ5wqlKThDZl1RRFYXyrw8DE3AUR3rW1n/tkMqRiWaNqcLqbilZ7Q7xWU4BIjoNOG2WWJp8nvfB6oRUC74RdFiD7F/UFgJDSmwxh5I5zIbv2iOvkl3MvYMS9NTIk2nqN8VEtREwBnRo19AlO0UAoGv3beLLNogYlwLyGmzfM74MPvcrqFuKZjS6sQR7nC5+ro11NRMv4UiniAbFQLffgvzCedh60XJOU1/LoBekQUhbTN3yxELZnYPa9601S/zwcdIwsd0Y3ChqOOBTVMVOwmRxTEixhTKOWCWYK2zxewFyx3JOoO8ERWBE

In [9]:
config: RunnableConfig = {"configurable": {"thread_id": "t4"}}

async for event in graph_1.astream_events({"messages": [{"role": "user", "content": "my name is krish"}]}, config=config, stream_mode="updates"):
    print(event)

{'event': 'on_chain_start', 'data': {'input': {'messages': [{'role': 'user', 'content': 'my name is krish'}]}}, 'name': 'LangGraph', 'tags': [], 'run_id': '01a1515d-a7bf-73a2-8caf-df052892a687', 'metadata': {'thread_id': 't4', 'ls_integration': 'langgraph'}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='my name is krish', additional_kwargs={}, response_metadata={}, id='b9dadadb-7bc6-4ea1-9374-6e04d51906eb')]}}, 'name': 'super_bot', 'tags': ['graph:step:1'], 'run_id': '01a1515d-a7c4-7331-9cbc-f5045eb3faa1', 'metadata': {'thread_id': 't4', 'ls_integration': 'langgraph', 'langgraph_step': 1, 'langgraph_node': 'super_bot', 'langgraph_triggers': ('branch:to:super_bot',), 'langgraph_path': ('__pregel_pull', 'super_bot'), 'langgraph_checkpoint_ns': 'super_bot:3a691cf0-5d55-03f5-f4da-8651e5b667df'}, 'parent_ids': ['01a1515d-a7bf-73a2-8caf-df052892a687']}
{'event': 'on_chain_stream', 'run_id': '01a1515d-a7c4-7331-9cbc-f5045eb3faa1', 'name': 